In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime

# Local project utilities
import sys
sys.path.append("..")
from parcel_calculations import add_improvement_ratio_fields
from cloud_utils import get_feature_data_with_geometry, ensure_geodataframe

# Config
SCRAPE_DATA = 0  # set to 1 to rescrape from ArcGIS
DATA_DIR = "data/spokane"
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
if SCRAPE_DATA == 1:
    base_url = "https://services1.arcgis.com/ozNll27nt9ZtPWOn/ArcGIS/rest/services/"
    parcel_gdf = get_feature_data_with_geometry("Parcels", base_url)

    today_str = datetime.now().strftime("%Y_%m_%d")
    out_path = os.path.join(DATA_DIR, f"spokane_parcels_{today_str}.parquet")
    parcel_gdf.to_parquet(out_path, index=False)
    print(f"✅ Saved new scrape to {out_path}")

else:
    files = glob.glob(os.path.join(DATA_DIR, "spokane_parcels_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parcel files found in {DATA_DIR}. Set SCRAPE_DATA=1 to scrape.")

    files_sorted = sorted(
        files,
        key=lambda x: datetime.strptime(
            os.path.basename(x).replace("spokane_parcels_", "").replace(".parquet", ""),
            "%Y_%m_%d"
        ),
        reverse=True
    )
    latest_file = files_sorted[0]
    print(f"✅ Loading most recent scrape: {latest_file}")
    parcel_gdf = pd.read_parquet(latest_file)

# Ensure GeoDataFrame
parcel_gdf = ensure_geodataframe(parcel_gdf)
print(f"✅ Loaded as {type(parcel_gdf).__name__} | CRS = {parcel_gdf.crs}")


In [ ]:
pd.set_option('display.max_columns', None)
display(parcel_gdf.head())


In [ ]:
# Restrict to parcels in the specified tax code areas with 'Spokane General' levy
spokane_general_tax_codes = [
    "0010", "0011", "0012", "0013", "0014", "0015", "0016", "0017",
    "0030", "0031", "0040", "0050", "0055", "0057", "0070", "0075", "0076",
    "0085", "0086"
]

# Use the column name that fits (likely 'tax_code_area' or close variant)
# Handle numeric tax_code_area by converting to string just in case
if "tax_code_area" in parcel_gdf.columns:
    parcel_gdf = parcel_gdf[parcel_gdf["tax_code_area"].astype(str).isin(spokane_general_tax_codes)].copy()
else:
    raise KeyError("Expected a 'tax_code_area' column in parcel_gdf.")

print(f"✅ Filtered to {len(parcel_gdf):,} parcels in 'Spokane General' levy areas")


In [ ]:
def categorize_property_type(prop_use_desc):
    # Direct mapping based on the property use descriptions from the data
    category_mapping = {
        "Single Family": ["Single Unit"],
        "Small Multi-Family (2-4 units)": ["Two-to-Four Unit"],
        "Large Multi-Family (5+ units)": ["Five-Plus Unit"],
        "Other Residential": ["Other Residential", "Vacation Home"],
        "Mobile Home Park": ["Mobile Home Park"],
        "Vacant Land": ["Vacant Land"],
        "Agricultural": ["Cur - Use - Ag", "Agricultural Not Classified", "Agricultural"],
        "Retail/Service/Commercial": [
            "Retail - General Mrchds", "Retail - Other", "Retail - Hardware", "Retail - Food", "Retail - Eating",
            "Retail - Auto", "Retail - Furniture", "Service - Finance", "Service - Professional", "Service - Repair",
            "Service - Education", "Service - Governmental", "Service - Construction", "Service - Personal",
            "Service - Business", "Wholesale", "Hotel/Motel", "Hotel/Condo", "Inst Lodging", "Recreational",
            "Resort - Camping", "Public Assembly", "Churches", "Park", "Other Cultural"
        ],
        "Manufacturing/Industrial": [
            "Manf - Other", "Manf - Fabricated Material", "Manf - Petroleum", "Manf - Printed Material",
            "Manf - Stone/Glass", "Manf - Printing", "Manf - Instrumentation", "Manf - Leather", "Manf - Paper",
            "Manufacturing - Food", "Manufacturing - Lumber", "Mining", "Utilities", "Communication"
        ],
        "Transportation - Parking": ["Trans - Parking"],
        "Transportation/Other": [
            "Trans - Highway", "Trans - Railroad", "Trans - Aircraft", "Trans - Motor", "Trans - Other"
        ],
        "Designated Forest": ["Designated Forest Lnd"],
        "Water Areas": ["Water Area"],
        "Marijuana": ["Marijuana Growing"],
        "Current Use Open": ["Cur - Use - Open"]
    }

    # Check for exact matches first
    for category, keywords in category_mapping.items():
        if prop_use_desc in keywords:
            return category

    # If no match found, return "Other"
    return "Other"

# Apply the function to the DataFrame
parcel_gdf['PROPERTY_CATEGORY'] = parcel_gdf['prop_use_desc'].apply(categorize_property_type)

In [ ]:
# -----------------------------
# 1) Clone dataframe
# -----------------------------
export_gdf = parcel_gdf.copy()

# -----------------------------
# 2) Ensure exemption flag exists
# -----------------------------
if "full_exmp" not in export_gdf.columns:
    if "taxable_amt" in export_gdf.columns:
        export_gdf["full_exmp"] = (export_gdf["taxable_amt"] <= 0).astype(int)
    else:
        raise ValueError("Need either 'full_exmp' or 'taxable_amt' to define exemptions.")

export_gdf["exemption_flag"] = (export_gdf["full_exmp"] == 1).astype(int)

# -----------------------------
# 3) Ensure improvement_value exists
# -----------------------------
if "improvement_value" not in export_gdf.columns:
    if "assessed_amt" in export_gdf.columns and "land_value" in export_gdf.columns:
        export_gdf["improvement_value"] = (export_gdf["assessed_amt"] - export_gdf["land_value"]).clip(lower=0)
    else:
        raise ValueError("Need 'assessed_amt' and 'land_value' to compute improvement_value.")

# -----------------------------
# 4) Ensure PROPERTY_CATEGORY exists
# -----------------------------

export_gdf["property_land_use_category"] = export_gdf["PROPERTY_CATEGORY"]

# -----------------------------
# 5) Refined land use classification
# -----------------------------
def categorize_property_refined(row):
    cat = str(row["PROPERTY_CATEGORY"])
    if "Vacant" in cat:
        return "Vacant"
    elif "Parking" in cat:
        return "Parking Lot"
    elif row["improvement_value"] < 0.5 * (row["land_value"] + row["improvement_value"]):
        return "Underdeveloped"
    else:
        return None

export_gdf["property_land_use_refined"] = export_gdf.apply(categorize_property_refined, axis=1)

# -----------------------------
# 6) Compute parcel area sqft
# -----------------------------
if "Shape__Area" in export_gdf.columns:
    # Your code assumed Shape__Area is m² (converts to sqft)
    export_gdf["area_sqft"] = export_gdf["Shape__Area"] * 10.7639
else:
    export_gdf["area_sqft"] = export_gdf.geometry.area * 10.7639

export_gdf["area_sqft"] = export_gdf["area_sqft"].replace(0, np.nan)

# -----------------------------
# 7) (Skipped current_tax, as instructed)
# -----------------------------
# Don't add or export current_tax

# -----------------------------
# 8) Per sqft metrics and full market value per sqft
# -----------------------------
# Full market value is assessed_amt if present
if "assessed_amt" in export_gdf.columns:
    export_gdf["full_market_value"] = export_gdf["assessed_amt"]
else:
    export_gdf["full_market_value"] = export_gdf.get("land_value", 0) + export_gdf.get("improvement_value", 0)

export_gdf["full_market_value_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]

export_gdf["land_value_per_sqft"] = export_gdf["land_value"] / export_gdf["area_sqft"]
export_gdf["improvement_value_per_sqft"] = export_gdf["improvement_value"] / export_gdf["area_sqft"]

# -----------------------------
# 8b) Derived improvement/land ratios
# -----------------------------
export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col="land_value",
    improvement_col="improvement_value"
)

# -----------------------------
# Save the link, ensuring it's present in the export data
# -----------------------------
if "link" not in export_gdf.columns:
    if "PID_NUM" in export_gdf.columns:
        export_gdf["link"] = export_gdf["PID_NUM"].astype(str).apply(
            lambda pid: f"https://cp.spokanecounty.org/SCOUT/propertyinformation/Summary.aspx?PID={pid}"
        )
    else:
        export_gdf["link"] = np.nan

# -----------------------------
# 9) Select columns (no current_tax), now including link
# -----------------------------
columns_to_export = [
    "geometry",
    "exemption_flag",
    "property_land_use_category",
    "property_land_use_refined",
    "full_market_value",
    "full_market_value_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "TLLDIMPROV",
    "IMPR_LAND_RATIO",
    "IMPR_LAND_PCT",
    "IMPR_PCT_TOTAL",
    "link"
]

export_final = export_gdf[columns_to_export].rename(columns={
    "land_value": "current_full_land_value"
})

# Ensure geometry validity
export_final["geometry"] = export_final["geometry"].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

# Ensure CRS is EPSG:4326
export_final = gpd.GeoDataFrame(export_final, geometry="geometry", crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs("EPSG:4326")
    print("✅ Converted to EPSG:4326")

# -----------------------------
# 10) Save Parquet: both canonical and dated version
# -----------------------------
canonical_path = os.path.join(DATA_DIR, "spokane-wa-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = os.path.join(DATA_DIR, f"spokane-wa-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print("Export columns:", export_final.columns.tolist())
print("\nRefined category counts:")
print(export_final["property_land_use_refined"].value_counts(dropna=False))


In [ ]:
# Optional: upload export_final to dev Azure blob
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "spokane-wa-parcels.parquet"
    local_path = os.path.join(DATA_DIR, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local parquet not found: {local_path}")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {local_path} -> {container}/{blob_name}")
else:
    print("upload_dev is False; skipping upload.")
